In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import celltypist
from celltypist import models
from celltypist.models import Model

# MASLD_pre celltypist analysis

In [ ]:
filtered_adata = scv.read('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_0102_hm.h5ad')
filtered_adata

In [ ]:
adata_pre=sc.read('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_0102.h5ad')

In [ ]:
adata_pre

In [ ]:
# 정규화: 각 세포의 총 발현을 1로 맞추고 로그 변환
sc.pp.normalize_total(adata_pre, target_sum=1e4)
sc.pp.log1p(adata_pre)

In [ ]:
celltypist.models.models_description()

In [ ]:
model_liver = models.Model.load(model = 'Healthy_Human_Liver.pkl')
model_liver.cell_types

In [ ]:
adata_pre

In [ ]:
adata_pre.obs_names_make_unique()

In [ ]:
# Healthy_Human_Liver.pkl
predictions = celltypist.annotate(adata_pre, model = 'Healthy_Human_Liver.pkl', majority_voting = True)

In [ ]:
# Healthy_Human_Liver.pkl
predictions = celltypist.annotate(adata_pre, model = 'Immune_All_Low.pkl', majority_voting = True)

In [ ]:
predictions.predicted_labels

In [ ]:
adata_pre = predictions.to_adata()

In [ ]:
adata_pre.obs

In [ ]:
filtered_adata.obs['celltypist_annot'] = adata_pre.obs['majority_voting']

In [ ]:
filtered_adata.obs['celltypist_immune_annot'] = adata_pre.obs['majority_voting']

In [ ]:
filtered_adata.obs

In [ ]:
sc.pl.umap(filtered_adata, color = 'celltypist_annot')

In [ ]:
sc.pl.umap(filtered_adata, color = 'celltypist_immune_annot')

In [ ]:
sc.pl.tsne(filtered_adata, color = 'celltypist_annot')

In [ ]:
sc.pl.tsne(filtered_adata, color = 'celltypist_immune_annot')

In [ ]:
filtered_adata.uns['cellannot_colors']=['#1f77b4','#17becf', '#279e68','#aa40fc', '#d62728', '#ff7f0e',
       '#8c564b', '#8c564b','#ff7f0e', '#aec7e8', '#ffbb78', '#98df8a'] 

In [ ]:
filtered_adata

In [ ]:
sc.pl.umap(filtered_adata, color = 'cellannot')

In [ ]:
filtered_adata.write('/home/jaehyunchoi/MASLD_collaboration/adata/sn_MASLD_annotation.h5ad')

In [ ]:
filtered_adata.obs_names_make_unique()

In [ ]:
filtered_adata.obs['cellannot_final'] = filtered_adata.obs['cellannot']

cluster2annotation = {
    "hepatocyte": "Hepatocytes",
    "MP": "Macrophage",
    "B-cell": "B cells",
    "T-cells&ILC": "T cells & ILC",
    "Endothelia": "Endothelia",
    "Mesenchyme": "Mesenchyme",
    "Cholangiocytes": "Cholangiocytes"
}

filtered_adata.uns['cellannot_final_colors']=['#1f77b4','#17becf', '#279e68','#aa40fc', '#d62728', '#ff7f0e',
       '#8c564b', '#8c564b','#ff7f0e', '#aec7e8', '#ffbb78', '#98df8a'] 

# add a new `.obs` column called `cell type` by mapping clusters to annotation using pandas `map` function
filtered_adata.obs['cellannot_final'] = filtered_adata.obs['cellannot_final'].map(cluster2annotation).astype("category")
sc.pl.umap(filtered_adata, color=['cellannot_final'])

In [ ]:
sc.pl.umap(filtered_adata, color=['SPI1','CSF1R'])

In [ ]:
sc.tl.rank_genes_groups(filtered_adata, groupby='cellannot_final', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(filtered_adata, n_genes=10)

In [ ]:
rank_df = sc.get.rank_genes_groups_df(filtered_adata, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    filtered_adata,
    var_names=top_genes,
    groupby="cellannot_final",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=True,
    show=True,
    figsize=(10, 7)
)

In [ ]:
sc.pl.heatmap(
    filtered_adata,
    var_names=top_genes,
    groupby="cellannot_final",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
import matplotlib.pyplot as plt

# heatmap 그리기 (show=False로 저장만 하고 안 보여줌)
sc.pl.heatmap(
    filtered_adata,
    var_names=top_genes,
    groupby="cellannot_final",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=False,
    #figsize=(10, 7)
    show_gene_labels=False
)

# 현재 figure 저장
fig = plt.gcf()
fig.savefig("/home/jaehyunchoi/MASLD_collaboration/analysis_notbook/analysis/Figure_pdf/Figure/sn_MASLD_heatmap.pdf", bbox_inches='tight', dpi=300)
plt.close(fig)

In [ ]:
# 그룹별 샘플링 수
n_cells_per_group = 100
groups = filtered_adata.obs['cellannot_final'].unique()

# 샘플링된 셀들의 인덱스 저장
selected_indices = []

for group in groups:
    group_cells = filtered_adata.obs[filtered_adata.obs['cellannot_final'] == group].index
    n = min(len(group_cells), n_cells_per_group)  # 그룹 크기가 작을 경우 자동 조절
    sampled = np.random.choice(group_cells, n, replace=False)
    selected_indices.extend(sampled)

adata_sampled = filtered_adata[selected_indices]

In [ ]:
fig_dict = sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="cellannot_final",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False,
    return_fig=True
)

print(type(fig_dict))
print(fig_dict.keys())

In [ ]:
ax = fig_dict['heatmap_ax']
fig = ax.get_figure()

fig.savefig("/home/jaehyunchoi/MASLD_collaboration/analysis_notbook/analysis/Figure_pdf/Figure/sn_MASLD_heatmap_1.pdf", bbox_inches='tight', dpi=300)
plt.close(fig)

In [ ]:
filtered_adata.write('/home/jaehyunchoi/MASLD_collaboration/adata/sn_MASLD_annotation.h5ad')

# Macrophage analysis

In [ ]:
adata_pre=sc.read('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_merged_0102.h5ad')

In [ ]:
adata_pre.obs['cellannot'] = filtered_adata.obs['cellannot_final']

In [ ]:
adata_mp = adata_pre[adata_pre.obs['cellannot'].isin(['Macrophage'])].copy()
adata_mp

In [ ]:
adata_mp.write('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_nuc_mp001.h5ad')

In [ ]:
sc.pp.normalize_total(adata_mp, target_sum=1e4)
sc.pp.log1p(adata_mp)

In [ ]:
# 변수 유전자 설정 
sc.pp.highly_variable_genes(adata_mp, flavor='seurat', n_top_genes=2000)
# Scale data
sc.pp.scale(adata_mp, max_value=10)
# Run PCA
sc.tl.pca(adata_mp, svd_solver='arpack')
print(adata_mp.obsm['X_pca'][:5, :5])
sc.pl.pca(adata_mp, color='sample')

import matplotlib.pyplot as plt
var_ratio = adata_mp.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(adata_mp, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(adata_mp, n_pcs=30, use_rep='X_pca')
sc.tl.umap(adata_mp)
sc.pl.umap(adata_mp, color=['sample'])

In [ ]:
adata_mp

In [ ]:
sc.pl.umap(adata_mp, color=['batch'])

In [ ]:
adata_mp.obs['sample'].value_counts()

In [ ]:
adata_mp.obs['NAS_state'] = adata_mp.obs['sample']
cluster2annotation = {
    "pati5_2": "MASLD",
    "pati5_3": "NAS",
    "pati5_5": "MASLD",
    "pati5_6": "NAS",
    "pati5_10": "NAS",
    "pati5_11": "NAS",
    "norm4_3": "Normal",
    "norm4_9": "Normal",
    "norm4_13": "Normal",
    "norm4_88": "Normal",
    "norm4_89": "Normal",
    "norm4_1": "Normal"
}

# add a new `.obs` column called `cell type` by mapping clusters to annotation using pandas `map` function
adata_mp.obs['NAS_state'] = adata_mp.obs['NAS_state'].map(cluster2annotation).astype("category")
sc.pl.umap(adata_mp, color=['NAS_state'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_mp.obsm['X_pca'],adata_mp.obs, 'sample')

# Harmony 결과를 저장
adata_mp.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_mp, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_mp)
sc.pl.umap(adata_mp, color=['NAS_state'])

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 차원 축소된 데이터 (예: PCA or UMAP 등)
X = adata_mp.obsm['X_pca']  # 또는 'X_umap' 등

inertias = []
k_range = range(2, 20)  

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)  # inertia = 클러스터 내부 거리 제곱합

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, inertias, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

In [ ]:
sc.tl.leiden(adata_mp, resolution=0.5, key_added="10cluster")
sc.pl.umap(adata_mp, color=['10cluster'])

In [ ]:
sc.tl.leiden(adata_mp, resolution=0.6, key_added="12cluster")
sc.pl.umap(adata_mp, color=['12cluster'])

In [ ]:
sc.pl.umap(adata_mp, color=['NAS_state'])

In [ ]:
sc.pl.umap(adata_mp, color=['SLC7A5'])

In [ ]:
adata_mp.obs

In [ ]:
cluster_labels = adata_mp.obs['10cluster']
batch_labels = adata_mp.obs['batch']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

crosstab.plot(kind='bar', stacked=True, figsize=(3, 7))
plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:

from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = adata_mp.obs['batch']
cluster = adata_mp.obs['10cluster']

# 클러스터 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters, key=int):  # 클러스터 번호 순 정렬
    # 각 조건별 개수 계산
    donor_in = ((state == "normal") & (cluster == clus)).sum()
    donor_out = ((state == "normal") & (cluster != clus)).sum()
    ipf_in = ((state == "MASLD") & (cluster == clus)).sum()
    ipf_out = ((state == "MASLD") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [ipf_in, ipf_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out)
    ipf_frac = ipf_in / (ipf_in + ipf_out)
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "ipf_fraction": ipf_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 정렬
df_result = df_result.sort_values("p_value")
print(df_result)
# cluster 열을 숫자형으로 바꾼 뒤 정렬
df_result['cluster'] = df_result['cluster'].astype(int)
df_result = df_result.sort_values('cluster')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='normal')
plt.bar(x + width/2, df_result['ipf_fraction'], width, label='MASLD')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('T cell cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['ipf_fraction'][i])
        plt.text(x[i], max_height + 0.005, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
p3lt.show()

In [ ]:
# 'EMR1'(ADGRE1),'MRC1','CD163','CD68','MARCO' 
sc.pl.dotplot(adata_mp,['ADGRE1','MRC1','CD163','CD68','MARCO','CD14','FCGR3A','CCR2','SLC7A5' ], groupby="10cluster", standard_scale="var")

In [ ]:
sc.pl.dotplot(adata_mp,['CLEC4F','TIMD4','MRC1','ESAM','LYVE1','CD36'], groupby="10cluster", standard_scale="var")

In [ ]:
sc.pl.umap(adata_mp, color=['ADGRE1','MRC1','CD163','CD14'])

In [ ]:
sc.pl.umap(adata_mp, color=['CLEC4F','TIMD4','MRC1','ESAM'])

In [ ]:
sc.pl.umap(adata_mp, color=['CD14','FCGR3A','CCR2'])

In [ ]:
sc.pl.umap(adata_mp, color=['CLEC4F','MKI67'])

In [ ]:
sc.pl.umap(adata_mp, color=['SPI1','CEBPB','CSF1R','KLF4'])

In [ ]:
sc.pl.umap(adata_mp, color=['CEBPB'])

In [ ]:
sc.tl.rank_genes_groups(adata_mp, groupby='10cluster', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_mp, n_genes=10)

In [ ]:
rank_df = sc.get.rank_genes_groups_df(adata_mp, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_mp,
    var_names=top_genes,
    groupby="10cluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=True,
    show=True,
    figsize=(10, 7)
)

In [ ]:
sc.pl.heatmap(
    adata_mp,
    var_names=top_genes,
    groupby="10cluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

In [ ]:
adata_mp_pre=sc.read('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_nuc_mp001.h5ad')
adata_mp_pre.obs_names_make_unique()
sc.pp.normalize_total(adata_mp_pre, target_sum=1e4)
sc.pp.log1p(adata_mp_pre)

In [ ]:
model_liver = models.Model.load(model = 'Immune_All_Low.pkl') # Immune_All_High.pkl
model_liver.cell_types

In [ ]:
# Healthy_Human_Liver.pkl
predictions = celltypist.annotate(adata_mp_pre, model = 'Immune_All_Low.pkl', majority_voting = True)
predictions.predicted_labels
adata_mp_pre = predictions.to_adata()

In [ ]:
# Healthy_Human_Liver.pkl
predictions = celltypist.annotate(adata_mp_pre, model = 'Immune_All_High.pkl', majority_voting = True)
predictions.predicted_labels
adata_mp_pre = predictions.to_adata()

In [ ]:
adata_mp_pre.obs

In [ ]:
adata_mp.obs['celltypist_immune'] = adata_mp_pre.obs['majority_voting']

In [ ]:
adata_mp.obs['celltypist_immune_high'] = adata_mp_pre.obs['majority_voting']

In [ ]:
sc.pl.umap(adata_mp, color=['celltypist_immune_high'])

In [ ]:
sc.pl.umap(adata_mp, color=['celltypist_immune'])

In [ ]:
sc.pl.umap(adata_mp, color=['10cluster'])

In [ ]:
sc.pl.tsne(adata_mp, color=['celltypist_immune_high'])

In [ ]:
sc.pl.tsne(adata_mp, color=['celltypist_immune'])

In [ ]:
sc.pl.tsne(adata_mp, color=['10cluster'])

In [ ]:
adata_mp

In [ ]:
ct = pd.crosstab(adata_mp.obs['celltypist_immune'],adata_mp.obs['celltypist_immune_high'])

In [ ]:
ct

In [ ]:
adata_mp.write('/home/jaehyunchoi/MASLD_collaboration/adata/MASLD_nuc_mp002.h5ad')

# celltypist macrophage sorting

In [ ]:
adata_mp